In [ ]:
# ================== 0. Setup & imports ==================
from google.colab import files
uploaded = files.upload()  # upload data_th_sp.csv from your PC

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor  # pip install xgboost if needed

In [ ]:
# ================== 1. Load data ==================
df = pd.read_csv("data_th_sp.csv")  # make sure this name matches your file

X    = df[["temp", "loading", "conc"]]
y_th = df["thcond"]
y_sp = df["spheat"]

X_train, X_test, y_th_train, y_th_test = train_test_split(
    X, y_th, test_size=0.2, random_state=42
)
X_train2, X_test2, y_sp_train, y_sp_test = train_test_split(
    X, y_sp, test_size=0.2, random_state=42
)

# ================== 2. Helpers ==================
def metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return r2, mae, rmse

def print_metrics(name, target, y_true, y_pred):
    r2, mae, rmse = metrics(y_true, y_pred)
    print(f"{name:22s} - {target}: R^2={r2:.4f}, MAE={mae:.4f}, RMSE={rmse:.4f}")

# formulas for linear / ridge
def print_linear_like_formula(model, target_name, label):
    coef = model.coef_
    inter = model.intercept_
    print(f"\n{label} formula for {target_name}:")
    print(
        f"{target_name} = {inter:.6f}"
        f" + ({coef[0]:.6f})*temp"
        f" + ({coef[1]:.6f})*loading"
        f" + ({coef[2]:.6f})*conc"
    )

# formula for polynomial
def print_poly_formula(pipe, target_name, degree):
    poly_step = pipe.named_steps["poly"]
    lin_step  = pipe.named_steps["lin"]
    names = poly_step.get_feature_names_out(["temp", "loading", "conc"])
    coefs = lin_step.coef_
    inter = lin_step.intercept_
    print(f"\nPolynomial (deg={degree}) formula for {target_name}:")
    print(f"{target_name} = {inter:.6f}", end="")
    # names[0] is "1" (bias term), skip it
    for name, c in zip(names[1:], coefs[1:]):
        print(f" + ({c:.6f})*{name}", end="")
    print()

# model list for generic training
def make_models():
    poly_degree = 2
    return [
        ("Linear Regression",
         LinearRegression()),
        (f"Polynomial Regression (deg={poly_degree})",
         Pipeline([
             ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
             ("lin", LinearRegression())
         ])),
        ("Random Forest",
         RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
        ("Gradient Boosting",
         GradientBoostingRegressor(
             n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)),
        ("XGBoost",
         XGBRegressor(
             n_estimators=300, learning_rate=0.05, max_depth=4,
             subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1)),
        ("Ridge Regression",
         Ridge(alpha=1.0)),
        ("KNN",
         KNeighborsRegressor(n_neighbors=5, weights="distance")),
        ("Neural Network",
         MLPRegressor(
             hidden_layer_sizes=(64, 64), activation="relu",
             learning_rate_init=0.001, max_iter=500, random_state=42)),
    ]

# ================== 3. Train all 8 models & metrics ==================
print("=== Results for thcond ===")
models_th = make_models()
for name, model in models_th:
    model.fit(X_train, y_th_train)
    y_pred = model.predict(X_test)
    print_metrics(name, "thcond", y_th_test, y_pred)

print("\n=== Results for spheat ===")
models_sp = make_models()
for name, model in models_sp:
    model.fit(X_train2, y_sp_train)
    y_pred = model.predict(X_test2)
    print_metrics(name, "spheat", y_sp_test, y_pred)

# ================== 4. Explicit formulas ==================
# Linear
lin_th = next(m for n, m in models_th if n == "Linear Regression")
lin_sp = next(m for n, m in models_sp if n == "Linear Regression")
print_linear_like_formula(lin_th, "thcond", "Linear")
print_linear_like_formula(lin_sp, "spheat", "Linear")

# Polynomial (degree 2)
poly_name = "Polynomial Regression (deg=2)"
poly_th = next(m for n, m in models_th if n == poly_name)
poly_sp = next(m for n, m in models_sp if n == poly_name)
print_poly_formula(poly_th, "thcond", degree=2)
print_poly_formula(poly_sp, "spheat", degree=2)

# Ridge
ridge_th = next(m for n, m in models_th if n == "Ridge Regression")
ridge_sp = next(m for n, m in models_sp if n == "Ridge Regression")
print_linear_like_formula(ridge_th, "thcond", "Ridge")
print_linear_like_formula(ridge_sp, "spheat", "Ridge")



In [ ]:
# ================== 5. Predictions from all models ==================
new_data = pd.DataFrame({
    "temp":   [55.0],
    "loading":[0.3],
    "conc":   [0.8]
})

print("\n=== Predictions for thcond ===")
for name, model in models_th:
    pred = float(model.predict(new_data)[0])
    print(f"{name:25s} -> thcond = {pred:.4f}")

print("\n=== Predictions for spheat ===")
for name, model in models_sp:
    pred = float(model.predict(new_data)[0])
    print(f"{name:25s} -> spheat = {pred:.4f}")
